In [16]:
import pandas as pd
import numpy as np

Loading the four cleaned datasets

In [17]:
from pathlib import Path

project_root = Path.cwd().parent.parent
processed = project_root / "data" / "processed"

df_bill = pd.read_csv(processed / "cleaned_billings.csv", low_memory=False)
df_ren  = pd.read_csv(processed / "cleaned_renewal_calls.csv", low_memory=False)
df_em   = pd.read_csv(processed / "cleaned_emails.csv", low_memory=False)
df_cc   = pd.read_csv(processed / "cleaned_cc_calls.csv", low_memory=False)

print(f"Billings:       {df_bill.shape}")
print(f"Renewal Calls:  {df_ren.shape}")
print(f"Emails:         {df_em.shape}")
print(f"CC Calls:       {df_cc.shape}")

Billings:       (122082, 53)
Renewal Calls:  (127307, 44)
Emails:         (123389, 27)
CC Calls:       (32882, 33)


### Merge Strategy

- **Billings** is the master table. Each row is a unique renewal event keyed by `(Co_Ref, Renewal_Year)`.
- **Emails** — all records are pre-renewal (prior_year / 45_out / 14_out / pre_renewal), so we aggregate the entire dataset by `(Co_Ref, year)`.
- **CC Calls** — we keep only calls where `Call_Date < Renewal_Month` (pre-renewal signals), then aggregate by `(Co_Ref, Call_Year)`.
- **Renewal Calls** — we keep only calls where `Call_Date > Renewal_Month` (post-renewal features), then aggregate by `(Co_Ref, Call_Year)`.

In [18]:
# Parse the date columns we need for temporal filtering
df_bill["Renewal_Month"] = pd.to_datetime(df_bill["Renewal_Month"], errors="coerce")
df_ren["Call_Date"] = pd.to_datetime(df_ren["Call_Date"], errors="coerce")
df_cc["Call_Date"] = pd.to_datetime(df_cc["Call_Date"], errors="coerce")

print("Date columns parsed.")

Date columns parsed.


## 1. Aggregating Emails

All email records are pre-renewal by nature (Time_to_Renewal is one of prior_year / 45_out / 14_out / pre_renewal),
so no date filtering is required. We aggregate by `(Co_Ref, year)` to match the billings granularity.

In [19]:
# Binary flag columns in emails — we want the proportion of Yes responses per customer-year
email_flag_cols = [
    "crm_accreditation_completed", "crm_timely_completion",
    "crm_progress_towards_accreditation", "crm_delays_in_accreditation",
    "crm_contractor_suggested_leave", "crm_contractor_engagement",
    "crm_dts_or_ssip_mentioned", "crm_customer_payment_intention",
    "crm_competitors_mentioned", "crm_platform_issues_raised",
    "crm_agent_chased_contractor", "crm_accreditation_issues",
    "crm_membership_overdue", "crm_dissatisified_with_renewal_price",
    "crm_customer_complained", "crm_refund_mentioned",
    "crm_negative_customer_experience", "crm_dissatisfaction_with_support",
    "crm_financial_hardship_mentioned"
]

# Convert flags to binary (1 = Yes, 0 = anything else) for aggregation
for col in email_flag_cols:
    df_em[col + "_flag"] = (df_em[col] == "Yes").astype(int)

# Build the aggregation dictionary
email_agg = {}

# Total email count
email_agg["em_email_count"] = ("crm_contractor_sentiment_score", "count")

# Mean of each binary flag (proportion of Yes)
for col in email_flag_cols:
    email_agg["em_" + col] = (col + "_flag", "mean")

# Sentiment score — mean and max
email_agg["em_sentiment_score_mean"] = ("crm_contractor_sentiment_score", "mean")
email_agg["em_sentiment_score_max"] = ("crm_contractor_sentiment_score", "max")

# Agent chase count — total and max
email_agg["em_agent_chase_total"] = ("crm_agent_chase_count", "sum")
email_agg["em_agent_chase_max"] = ("crm_agent_chase_count", "max")

# Auto renewal status — max (captures if any email flagged it)
email_agg["em_auto_renewal_status_max"] = ("crm_auto_renewal_status", "max")

df_email_agg = df_em.groupby(["Co_Ref", "year"]).agg(**email_agg).reset_index()

# Mode-based features
sentiment_mode = df_em.groupby(["Co_Ref", "year"])["crm_contractor_sentiment"].agg(
    lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else "Not Discussed"
).reset_index().rename(columns={"crm_contractor_sentiment": "em_sentiment_mode"})

membership_mode = df_em.groupby(["Co_Ref", "year"])["crm_membership_level"].agg(
    lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else "Not Discussed"
).reset_index().rename(columns={"crm_membership_level": "em_membership_level_mode"})

# Merge modes back
df_email_agg = df_email_agg.merge(sentiment_mode, on=["Co_Ref", "year"], how="left")
df_email_agg = df_email_agg.merge(membership_mode, on=["Co_Ref", "year"], how="left")

print(f"Email aggregated: {df_email_agg.shape}")
df_email_agg.head()

Email aggregated: (37964, 29)


,Co_Ref,year,em_email_count,em_crm_accreditation_completed,em_crm_timely_completion,em_crm_progress_towards_accreditation,em_crm_delays_in_accreditation,em_crm_contractor_suggested_leave,em_crm_contractor_engagement,em_crm_dts_or_ssip_mentioned,...,em_crm_negative_customer_experience,em_crm_dissatisfaction_with_support,em_crm_financial_hardship_mentioned,em_sentiment_score_mean,em_sentiment_score_max,em_agent_chase_total,em_agent_chase_max,em_auto_renewal_status_max,em_sentiment_mode,em_membership_level_mode
0,AA0584,2025,2,0.0,0.0,1.000000,0.500000,0.0,1.000000,0.500000,...,0.00,0.0,0.0,50.0,50.0,2,2,0,Neutral,In Progress
1,AA0641,2025,3,0.0,0.0,0.666667,0.666667,0.0,0.333333,0.333333,...,0.00,0.0,0.0,50.0,50.0,2,2,0,Not Discussed,In Progress
2,AA0750,2025,2,0.0,0.0,1.000000,0.500000,0.0,1.000000,0.500000,...,0.50,0.5,0.0,50.0,50.0,3,2,0,Neutral,In Progress
3,AA0784,2026,4,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,...,0.25,0.0,0.0,50.0,50.0,3,1,0,Not Discussed,In Progress
4,AA0794,2025,3,0.0,0.0,0.666667,0.666667,0.0,0.333333,0.666667,...,0.00,0.0,0.0,50.0,50.0,5,3,0,Not Discussed,In Progress


## 2. Aggregating Customer Care Calls (pre-renewal)

We only keep CC calls that happened **before** the customer's `Renewal_Month` to capture pre-renewal signals.

In [20]:
# Bring Renewal_Month into cc_calls for temporal filtering
bill_dates = df_bill[["Co_Ref", "Renewal_Year", "Renewal_Month"]].copy()

df_cc_merged = df_cc.merge(
    bill_dates,
    left_on=["Co_Ref", "Call_Year"],
    right_on=["Co_Ref", "Renewal_Year"],
    how="inner"
)

# Keep only calls BEFORE the renewal month
df_cc_pre = df_cc_merged[df_cc_merged["Call_Date"] < df_cc_merged["Renewal_Month"]].copy()
print(f"CC calls total: {len(df_cc)}, after inner join: {len(df_cc_merged)}, pre-renewal: {len(df_cc_pre)}")

CC calls total: 32882, after inner join: 31124, pre-renewal: 15081


In [21]:
# Binary flag columns in CC calls (cc_care_package is categorical, not a flag)
cc_flag_cols = [
    "cc_care_package_discussed", "cc_urgency_getting_on_site",
    "cc_external_consultant", "cc_agent_cross_sell_attempt",
    "cc_customer_issues_concerns", "cc_business_struggles_financial_hardship",
    "cc_questionnaire_completion", "cc_chasing_response",
    "cc_issues_within_questionnaire", "cc_login_issues", "cc_platform_issues",
    "cc_dissatisfaction_time_to_complete", "cc_process_complexity_concerns",
    "cc_questions_harder_than_expected", "cc_dissatisfaction_support",
    "cc_pricing_mentioned", "cc_refund_discussed",
    "cc_contractor_suggest_leave", "cc_contractor_complained",
    "cc_pricing_sentiment_impact"
]

for col in cc_flag_cols:
    if col in df_cc_pre.columns:
        df_cc_pre[col + "_flag"] = (df_cc_pre[col] == "Yes").astype(int)

# Build aggregation
cc_agg = {}
cc_agg["cc_call_count"] = ("Call_Date", "count")

for col in cc_flag_cols:
    if col + "_flag" in df_cc_pre.columns:
        cc_agg["cc_" + col.replace("cc_", "")] = (col + "_flag", "mean")

# Sentiment scores — averages
score_cols = [
    "cc_contractor_sentiment_start_score", "cc_contractor_sentiment_end_score",
    "cc_contractor_sentiment_overall_score", "cc_contractor_sentiment_issues_score"
]
for col in score_cols:
    if col in df_cc_pre.columns:
        short = col.replace("cc_contractor_sentiment_", "cc_sent_")
        cc_agg[short + "_mean"] = (col, "mean")

df_cc_agg = df_cc_pre.groupby(["Co_Ref", "Call_Year"]).agg(**cc_agg).reset_index()

# Sentiment mode
cc_sent_mode = df_cc_pre.groupby(["Co_Ref", "Call_Year"])["cc_contractor_sentiment"].agg(
    lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else "Not Discussed"
).reset_index().rename(columns={"cc_contractor_sentiment": "cc_sentiment_mode"})

df_cc_agg = df_cc_agg.merge(cc_sent_mode, on=["Co_Ref", "Call_Year"], how="left")

print(f"CC calls aggregated: {df_cc_agg.shape}")
df_cc_agg.head()

CC calls aggregated: (8406, 28)


,Co_Ref,Call_Year,cc_call_count,cc_care_package_discussed,cc_urgency_getting_on_site,cc_external_consultant,cc_agent_cross_sell_attempt,cc_customer_issues_concerns,cc_business_struggles_financial_hardship,cc_questionnaire_completion,...,cc_pricing_mentioned,cc_refund_discussed,cc_contractor_suggest_leave,cc_contractor_complained,cc_pricing_sentiment_impact,cc_sent_start_score_mean,cc_sent_end_score_mean,cc_sent_overall_score_mean,cc_sent_issues_score_mean,cc_sentiment_mode
0,AA0750,2025,1,0.0,0.000000,1.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,50.0,90.0,90.000000,90.0,Satisfied
1,AA0882,2025,1,0.0,0.000000,1.0,0.0,0.000000,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,50.0,80.0,70.000000,70.0,Neutral
2,AA2827,2025,3,0.0,0.333333,0.0,0.0,0.333333,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,50.0,80.0,73.333333,70.0,Neutral
3,AA3881,2025,2,0.0,0.500000,0.0,0.0,0.500000,0.5,0.0,...,0.0,0.0,0.0,0.5,0.0,35.0,70.0,70.000000,50.0,Dissatisfied
4,AA4063,2025,1,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,50.0,50.0,80.000000,90.0,Not Discussed


## 3. Aggregating Renewal Calls (post-renewal only)

We filter renewal calls to only those where `Call_Date > Renewal_Month` to capture post-renewal behaviour.

In [22]:
# Bring Renewal_Month into renewal calls for temporal filtering
df_ren_merged = df_ren.merge(
    bill_dates,
    left_on=["Co_Ref", "Call_Year"],
    right_on=["Co_Ref", "Renewal_Year"],
    how="inner"
)

# Keep only calls AFTER the renewal month
df_ren_post = df_ren_merged[df_ren_merged["Call_Date"] > df_ren_merged["Renewal_Month"]].copy()
print(f"Renewal calls total: {len(df_ren)}, after inner join: {len(df_ren_merged)}, post-renewal: {len(df_ren_post)}")

Renewal calls total: 127307, after inner join: 122429, post-renewal: 86155


In [23]:
# Binary flag columns in renewal calls
ren_flag_cols = [
    "Serious_Complaint", "Other_Complaint", "Discussion_on_Price_Increase",
    "Renewal_Impact_Due_to_Price_Increase", "Discount_or_Waiver_Requested",
    "Call_Reschedule_Request", "Agent_Flagged_Membership_Status_Alert",
    "Agent_Renewal_Initiation", "Explicit_Competitor_Mention",
    "Explicit_Switching_Intent", "Price_Switching_Mentioned",
    "Competitor_Value_Comparison", "Competitor_Benefits_Mentioned",
    "Customer_Asked_For_Justification", "Discount_Offered",
    "Has_Complaint", "Has_Negative_Reaction"
]

for col in ren_flag_cols:
    if col in df_ren_post.columns:
        df_ren_post[col + "_flag"] = (df_ren_post[col] == "Yes").astype(int)

# Build aggregation
ren_agg = {}
ren_agg["ren_call_count"] = ("Call_Date", "count")

for col in ren_flag_cols:
    if col + "_flag" in df_ren_post.columns:
        ren_agg["ren_" + col.lower()] = (col + "_flag", "mean")

# Score columns — averages
if "Call_Friction_Score" in df_ren_post.columns:
    ren_agg["ren_friction_score_mean"] = ("Call_Friction_Score", "mean")
    ren_agg["ren_friction_score_max"] = ("Call_Friction_Score", "max")

if "Call_Length_Score" in df_ren_post.columns:
    ren_agg["ren_call_length_mean"] = ("Call_Length_Score", "mean")

if "Call_Number" in df_ren_post.columns:
    ren_agg["ren_max_call_number"] = ("Call_Number", "max")

df_ren_agg = df_ren_post.groupby(["Co_Ref", "Call_Year"]).agg(**ren_agg).reset_index()

# Mode-based features for categorical columns
cat_ren_cols = [
    "Churn_Category", "Complaint_Category", "Customer_Reaction_Category",
    "Agent_Renewal_Pitch_Category", "Customer_Renewal_Response_Category",
    "Agent_Response_Category", "Membership_Renewal_Decision",
    "Customer_Response", "Justification_Category",
    "Reason_For_Renewal_Category", "Agent_Response_To_Cancel_Category",
    "Argument_That_Convinced_Customer_to_Stay_Category"
]

for cat_col in cat_ren_cols:
    if cat_col in df_ren_post.columns:
        mode_df = df_ren_post.groupby(["Co_Ref", "Call_Year"])[cat_col].agg(
            lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else "Unknown"
        ).reset_index().rename(columns={cat_col: "ren_" + cat_col.lower() + "_mode"})
        df_ren_agg = df_ren_agg.merge(mode_df, on=["Co_Ref", "Call_Year"], how="left")

print(f"Renewal calls aggregated: {df_ren_agg.shape}")
df_ren_agg.head()

Renewal calls aggregated: (34047, 36)


,Co_Ref,Call_Year,ren_call_count,ren_serious_complaint,ren_other_complaint,ren_discussion_on_price_increase,ren_renewal_impact_due_to_price_increase,ren_discount_or_waiver_requested,ren_call_reschedule_request,ren_agent_flagged_membership_status_alert,...,ren_customer_reaction_category_mode,ren_agent_renewal_pitch_category_mode,ren_customer_renewal_response_category_mode,ren_agent_response_category_mode,ren_membership_renewal_decision_mode,ren_customer_response_mode,ren_justification_category_mode,ren_reason_for_renewal_category_mode,ren_agent_response_to_cancel_category_mode,ren_argument_that_convinced_customer_to_stay_category_mode
0,AA0641,2024,1,0.0,0.000,0.0000,0.0,0.0,0.0,0.0,...,Not Mentioned,Not Mentioned,Not Mentioned,Not Mentioned,Pending/Unknown,Not Mentioned,Not Applicable,Not Applicable,Not Applicable,Not Applicable
1,AA0784,2026,3,0.0,0.000,0.0000,0.0,0.0,0.0,0.0,...,Not Mentioned,Not Mentioned,Communication,Customer Communication,Cancelled,Yes,Not Applicable,Not Applicable,Not Applicable,Not Applicable
2,AA0794,2025,6,0.0,0.000,0.0000,0.0,0.0,0.0,0.0,...,Not Mentioned,Discussion / Introduction / Inquiry,Not Mentioned,Not Mentioned,Cancelled,Not Mentioned,Not Applicable,Not Applicable,Not Applicable,Not Applicable
3,AA0994,2024,3,0.0,0.000,0.0000,0.0,0.0,0.0,0.0,...,Not Mentioned,Not Mentioned,Not Mentioned,Not Mentioned,Pending/Unknown,Not Mentioned,Not Applicable,Not Applicable,Not Applicable,Not Applicable
4,AA0994,2025,16,0.0,0.125,0.0625,0.0,0.0,0.0,0.0,...,Not Mentioned,Not Mentioned,Not Mentioned,Not Mentioned,Cancelled,Yes,Not Applicable,Not Applicable,Not Applicable,Not Applicable


## 4. Merging Everything onto Billings

Left join the 3 aggregated interaction tables onto the master billings table using `(Co_Ref, Renewal_Year)`.

In [24]:
master = df_bill.copy()
print(f"Starting shape: {master.shape}")

# Merge emails
master = master.merge(
    df_email_agg,
    left_on=["Co_Ref", "Renewal_Year"],
    right_on=["Co_Ref", "year"],
    how="left"
).drop(columns=["year"], errors="ignore")
print(f"After email merge: {master.shape}")

# Merge CC calls
master = master.merge(
    df_cc_agg,
    left_on=["Co_Ref", "Renewal_Year"],
    right_on=["Co_Ref", "Call_Year"],
    how="left"
).drop(columns=["Call_Year"], errors="ignore")
print(f"After CC merge: {master.shape}")

# Merge renewal calls
master = master.merge(
    df_ren_agg,
    left_on=["Co_Ref", "Renewal_Year"],
    right_on=["Co_Ref", "Call_Year"],
    how="left"
).drop(columns=["Call_Year"], errors="ignore")
print(f"After renewal merge: {master.shape}")

Starting shape: (122082, 53)
After email merge: (122082, 80)
After CC merge: (122082, 106)
After renewal merge: (122082, 140)


Filling nulls introduced by the left joins — customers without any interaction in a given dataset
get 0 for count/score columns and a descriptive default for categorical aggregates.

In [25]:
# Count columns — fill with 0 (no interactions)
count_cols = [c for c in master.columns if c.endswith("_count") or c.endswith("_total")]
for col in count_cols:
    master[col] = master[col].fillna(0).astype(int)

# Numeric score/proportion columns — fill with 0
num_fill = master.select_dtypes(include=["float64"]).columns
# Only fill the newly-added columns (em_, cc_, ren_ prefix)
new_num = [c for c in num_fill if c.startswith(("em_", "cc_", "ren_"))]
for col in new_num:
    master[col] = master[col].fillna(0)

# Categorical mode columns — fill with 'No Interaction'
mode_cols = [c for c in master.columns if c.endswith("_mode")]
for col in mode_cols:
    master[col] = master[col].fillna("No Interaction")

# em_membership_level_mode
if "em_membership_level_mode" in master.columns:
    master["em_membership_level_mode"] = master["em_membership_level_mode"].fillna("No Interaction")

# em_auto_renewal_status_max
if "em_auto_renewal_status_max" in master.columns:
    master["em_auto_renewal_status_max"] = master["em_auto_renewal_status_max"].fillna(0).astype(int)

# Remaining nulls summary
remaining = master.isnull().sum()
remaining = remaining[remaining > 0]
print(f"Columns with remaining nulls: {len(remaining)}")
if len(remaining) > 0:
    print(remaining.sort_values(ascending=False))

Columns with remaining nulls: 4
Last_Renewal         48291
Closed_Date           8188
Registration_Date     1018
Proforma_Date          304
dtype: int64


Final verification — confirm the row count matches billings and inspect the merged dataset

In [26]:
assert len(master) == len(df_bill), f"Row count mismatch: {len(master)} vs {len(df_bill)}"
print(f"Row count matches billings: {len(master)}")
print(f"Total columns: {len(master.columns)}")
print(f"\nNew feature columns added: {len(master.columns) - len(df_bill.columns)}")

Row count matches billings: 122082
Total columns: 140

New feature columns added: 87


In [27]:
master.dtypes

Co_Ref                                                                   str
Renewal_Month                                                 datetime64[us]
Sustainability_Score                                                 float64
Total_Renewal_Score_New                                              float64
Last_Years_Price                                                     float64
                                                                   ...      
ren_customer_response_mode                                               str
ren_justification_category_mode                                          str
ren_reason_for_renewal_category_mode                                     str
ren_agent_response_to_cancel_category_mode                               str
ren_argument_that_convinced_customer_to_stay_category_mode               str
Length: 140, dtype: object

In [28]:
master.info()

<class 'pandas.DataFrame'>
RangeIndex: 122082 entries, 0 to 122081
Columns: 140 entries, Co_Ref to ren_argument_that_convinced_customer_to_stay_category_mode
dtypes: datetime64[us](1), float64(86), int64(15), str(38)
memory usage: 130.4 MB


In [29]:
master.head()

,Co_Ref,Renewal_Month,Sustainability_Score,Total_Renewal_Score_New,Last_Years_Price,Auto_Renewal_Score,Status_Scores,Anchoring_Score,Tenure_Scores,Proforma_Auto_Renewal,...,ren_customer_reaction_category_mode,ren_agent_renewal_pitch_category_mode,ren_customer_renewal_response_category_mode,ren_agent_response_category_mode,ren_membership_renewal_decision_mode,ren_customer_response_mode,ren_justification_category_mode,ren_reason_for_renewal_category_mode,ren_agent_response_to_cancel_category_mode,ren_argument_that_convinced_customer_to_stay_category_mode
0,VT6174,2024-11-01,8.0,42.5,799.0,9,9,7.5,9.0,Yes,...,No Interaction,No Interaction,No Interaction,No Interaction,No Interaction,No Interaction,No Interaction,No Interaction,No Interaction,No Interaction
1,VD3828,2025-08-01,8.0,41.5,799.0,9,9,7.5,8.0,Yes,...,Not Mentioned,Not Mentioned,Not Mentioned,Not Mentioned,Pending/Unknown,Not Mentioned,Not Applicable,Not Applicable,Not Applicable,Not Applicable
2,DV8120,2025-03-01,8.0,33.0,799.0,8,0,7.5,9.5,Yes,...,Not Mentioned,Not Mentioned,Not Mentioned,Not Mentioned,Cancelled,Not Mentioned,Not Applicable,Not Applicable,Not Applicable,Not Applicable
3,EZ9894,2025-06-01,9.5,44.5,799.0,9,9,7.5,9.5,Yes,...,Not Mentioned,Not Mentioned,Not Mentioned,Not Mentioned,Pending/Unknown,Not Mentioned,Not Applicable,Not Applicable,Not Applicable,Not Applicable
4,FA8957,2025-03-01,9.5,42.5,799.0,9,8,7.5,8.5,Yes,...,No Interaction,No Interaction,No Interaction,No Interaction,No Interaction,No Interaction,No Interaction,No Interaction,No Interaction,No Interaction


## 4.5 Date Column Cleanup

We remove date columns that are not relevant for post-renewal churn calculation (`Last_Renewal`, `Registration_Date`, `Proforma_Date`) and use `Closed_Date` to create a useful post-renewal timeline feature.

In [30]:
import pandas as pd

# Convert dates to calculate timedelta
master['Closed_Date'] = pd.to_datetime(master['Closed_Date'], errors='coerce')
master['Prospect_Renewal_Date'] = pd.to_datetime(master['Prospect_Renewal_Date'], errors='coerce')

# Feature: How many days after the renewal date was the account finally closed/won?
master['Days_To_Close_Post_Renewal'] = (master['Closed_Date'] - master['Prospect_Renewal_Date']).dt.days

# Drop the unnecessary date columns
cols_to_drop = ['Last_Renewal', 'Registration_Date', 'Proforma_Date']
master = master.drop(columns=[c for c in cols_to_drop if c in master.columns], errors='ignore')

print(f"Created 'Days_To_Close_Post_Renewal'. Dropped: {cols_to_drop}")
print(f"New shape: {master.shape}")

Created 'Days_To_Close_Post_Renewal'. Dropped: ['Last_Renewal', 'Registration_Date', 'Proforma_Date']
New shape: (122082, 138)


Saving the master dataset

In [32]:
import os

out_dir = project_root / "data" / "processed"
os.makedirs(out_dir, exist_ok=True)

out_file = out_dir / "master_churn_dataset.csv"
master.to_csv(out_file, index=False)
print(f"Master dataset saved to {out_file}")
print(f"Final shape: {master.shape}")

Master dataset saved to c:\Users\MadhavGoyal\Desktop\DS Project'\Churn-Prediction\data\processed\master_churn_dataset.csv
Final shape: (122082, 138)
